In [2]:
import pandas as pd, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)

df = pd.read_csv("C:/Users/Elize/Documents/YR4 1st sem/APPLIED MULT DATA ANALYSIS/world-happiness-report-2021.csv", encoding="utf-8-sig")
vars_ = ["Ladder score","Logged GDP per capita","Social support","Healthy life expectancy",
         "Freedom to make life choices","Generosity","Perceptions of corruption"]
short = ["Ladder","LogGDP","Social","HLE","Freedom","Generosity","Corruption"]
X = df[vars_].copy(); X.columns = short

# ---- Part A ----
print("A1: n =", X.shape[0], " p =", X.shape[1], " missing =", int(X.isna().sum().sum()))
print(X.agg(["mean","std"]).T.round(3))
R = X.corr(); print(R.round(3))
# strongest pairs
pairs = (R.where(np.triu(np.ones(R.shape),1).astype(bool)).stack().sort_values(key=abs, ascending=False))
print(pairs.head(6).round(3))
fig, ax = plt.subplots(figsize=(6.5,5.5)); im = ax.imshow(R, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(7)); ax.set_xticklabels(short, rotation=45, ha="right"); ax.set_yticks(range(7)); ax.set_yticklabels(short)
for i in range(7):
    for j in range(7): ax.text(j,i,f"{R.iloc[i,j]:.3f}",ha="center",va="center",fontsize=7)
fig.colorbar(im); ax.set_title("Correlation Matrix"); fig.tight_layout(); fig.savefig("correlation_heatmap.png", dpi=200); plt.close()

# ---- Part B ----  (PCA on standardized data = eigen-decomposition of correlation matrix)
Z = (X - X.mean()) / X.std(ddof=1)
eigval, eigvec = np.linalg.eigh(R.values)
o = np.argsort(eigval)[::-1]; eigval, eigvec = eigval[o], eigvec[:, o]
# sign convention: make the largest-|loading| of each PC positive in PC1 only if it matches the "well-being" direction
if eigvec[0,0] < 0: eigvec[:,0] *= -1          # PC1: higher score = higher well-being
prop = eigval/eigval.sum(); cum = np.cumsum(prop)
tab = pd.DataFrame({"Eigenvalue":eigval,"Proportion":prop,"Cumulative":cum}, index=[f"PC{i}" for i in range(1,8)])
print(tab.round(3))
load = pd.DataFrame(eigvec, index=short, columns=[f"PC{i}" for i in range(1,8)])
print(load.round(3))
print("PC1 |coef| ranking:\n", load["PC1"].abs().sort_values(ascending=False).round(3))
scores = Z.values @ eigvec
sc = pd.DataFrame(scores[:, :2], columns=["PC1","PC2"]); sc.insert(0, "Country", df["Country name"])
print(sc.sort_values("PC1", ascending=False).head(5).round(3)); print(sc.sort_values("PC1").head(5).round(3))
print(sc.sort_values("PC2", ascending=False).head(5).round(3)); print(sc.sort_values("PC2").head(5).round(3))
# loadings (correlations of variables with PCs) for PC1-PC3
cl = load.iloc[:, :3] * np.sqrt(eigval[:3]); print(cl.round(3))
print("Philippines:", sc[sc.Country=="Philippines"].round(3).to_dict("records"))

# ---- Part C ----
fig, ax = plt.subplots(figsize=(6.5,4.2)); ax.plot(range(1,8), eigval, "o-", color="#1f4e79")
ax.axhline(1, ls="--", color="red", label="Kaiser line (eigenvalue = 1)")
for i,e in enumerate(eigval): ax.annotate(f"{e:.3f}", (i+1,e), textcoords="offset points", xytext=(6,6), fontsize=8)
ax.set_xlabel("Principal Component"); ax.set_ylabel("Eigenvalue"); ax.set_title("Scree Plot"); ax.legend(); fig.tight_layout(); fig.savefig("scree_plot.png", dpi=200); plt.close()
print("Kaiser retained:", int((eigval>1).sum()))

# ---- Part D ----
fig, ax = plt.subplots(figsize=(6.5,5.5))
ax.scatter(sc.PC1, sc.PC2, s=10, color="grey", alpha=.6)
for _, r in sc.iterrows():
    if r.Country in ["Finland","Denmark","Afghanistan","Philippines","Indonesia","Singapore","Lebanon","Zimbabwe","Bulgaria"]:
        ax.annotate(r.Country, (r.PC1, r.PC2), fontsize=7)
for v in short:
    ax.arrow(0,0,load.loc[v,"PC1"]*3,load.loc[v,"PC2"]*3,color="#c00000",head_width=.06)
    ax.text(load.loc[v,"PC1"]*3.2,load.loc[v,"PC2"]*3.2,v,color="#c00000",fontsize=8)
ax.axhline(0,lw=.5,color="k"); ax.axvline(0,lw=.5,color="k")
ax.set_xlabel(f"PC1 ({prop[0]*100:.1f}%)"); ax.set_ylabel(f"PC2 ({prop[1]*100:.1f}%)"); ax.set_title("PCA Biplot")
fig.tight_layout(); fig.savefig("biplot.png", dpi=200); plt.close()
tab.round(3).to_csv("eigen_table.csv"); load.round(3).to_csv("loadings.csv")

A1: n = 149  p = 7  missing = 0
              mean    std
Ladder       5.533  1.074
LogGDP       9.432  1.159
Social       0.815  0.115
HLE         64.993  6.762
Freedom      0.792  0.113
Generosity  -0.015  0.151
Corruption   0.727  0.179
            Ladder  LogGDP  Social    HLE  Freedom  Generosity  Corruption
Ladder       1.000   0.790   0.757  0.768    0.608      -0.018      -0.421
LogGDP       0.790   1.000   0.785  0.859    0.432      -0.199      -0.342
Social       0.757   0.785   1.000  0.723    0.483      -0.115      -0.203
HLE          0.768   0.859   0.723  1.000    0.461      -0.162      -0.364
Freedom      0.608   0.432   0.483  0.461    1.000       0.169      -0.401
Generosity  -0.018  -0.199  -0.115 -0.162    0.169       1.000      -0.164
Corruption  -0.421  -0.342  -0.203 -0.364   -0.401      -0.164       1.000
LogGDP  HLE       0.859
Ladder  LogGDP    0.790
LogGDP  Social    0.785
Ladder  HLE       0.768
        Social    0.757
Social  HLE       0.723
dtype: float64
 